In [1]:
import joblib
import shap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [9]:
X_train_scaled = pd.read_csv('../Data/Sampled_Data/X_train_scaled.csv')
X_test_scaled = pd.read_csv('../Data/Sampled_Data/X_test_scaled.csv')

X_train = pd.read_csv('../Data/Sampled_Data/X_train.csv')
X_test = pd.read_csv('../Data/Sampled_Data/X_test.csv')

y_train = pd.read_csv('../Data/Sampled_Data/y_train.csv').squeeze()
y_test = pd.read_csv('../Data/Sampled_Data/y_test.csv').squeeze()


print("Train shape:", X_train_scaled.shape)
print("Test shape:", X_test_scaled.shape)

y_train_bin = (y_train != 0).astype(int)
print(pd.Series(y_train_bin).value_counts())

y_test_bin = (y_test != 0).astype(int)
print(pd.Series(y_test_bin).value_counts())


Train shape: (2262300, 78)
Test shape: (565576, 78)
0
0    1817055
1     445245
Name: count, dtype: int64
0
0    454265
1    111311
Name: count, dtype: int64


In [10]:
rf = joblib.load('../Data/Sampled_Data/rf.pkl')

In [11]:
from sklearn.model_selection import train_test_split

X_sample, _, y_sample, _ = train_test_split(
    X_test,
    y_test,
    test_size=0.9,
    stratify=y_test,
    random_state=42
)


In [12]:
print(type(X_sample))


<class 'pandas.core.frame.DataFrame'>


In [13]:
from sklearn.utils import resample

X_sample_shap = resample(X_sample, n_samples=1500, random_state=42)

X_sample_shap = pd.DataFrame(
    X_sample_shap,
    columns=X_sample.columns
)


In [31]:
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_sample_shap)

In [42]:
print("Original SHAP shape:", shap_values.shape)

if len(shap_values.shape) == 3:
    shap_values = shap_values[:, :, 1]

print("After class selection:", shap_values.shape)

mean_abs = np.mean(np.abs(shap_values), axis=0)

rf_importance = pd.DataFrame({
    "feature": X_sample_shap.columns,
    "importance": mean_abs
}).sort_values("importance", ascending=False)

Original SHAP shape: (1500, 78)
After class selection: (1500, 78)


In [43]:
print(rf_importance.head(10))

   feature  importance
0        0    0.044086
5        5    0.034974
13      13    0.029874
65      65    0.025538
12      12    0.025083
52      52    0.024968
67      67    0.023184
42      42    0.022502
41      41    0.021803
54      54    0.021191


In [44]:
overall_magnitude = np.mean(np.abs(shap_values))
print("RF baseline SHAP magnitude:", overall_magnitude)

RF baseline SHAP magnitude: 0.007110072405893693


In [45]:
variance = np.var(np.abs(shap_values), axis=0)

rf_variance = pd.DataFrame({
    "feature": X_sample.columns,
    "variance": variance
}).sort_values("variance", ascending=False)

In [ ]:
rf_importance.to_csv("../Data/Sampled_Data/rf_baseline_shap.csv", index=False)
rf_variance.to_csv("../Data/Sampled_Data/rf_baseline_variance.csv", index=False)
del rf_importance
del rf_variance
del rf

In [48]:
rf_smote = joblib.load("../Data/Sampled_Data/rf_smote.pkl")